In [ ]:
import pandas                    as pd
import matplotlib.pyplot         as plt
import numpy                     as np
# Defining Primordial Functions
def adder(Data, times):
    
    for i in range(1, times + 1):
    
        new = np.zeros((len(Data), 1), dtype = float)
        Data = np.append(Data, new, axis = 1)        
    return Data
def deleter(Data, index, times):
    
    for i in range(1, times + 1):
    
        Data = np.delete(Data, index, axis = 1)        
    return Data
   
def jump(Data, jump):
    
    Data = Data[jump:, ]
    
    return Data
def indicator_plot_double(Data, first_panel, second_panel, window = 250):    
    fig, ax = plt.subplots(2, figsize = (10, 5))
    
    ax[0].plot(Data[-window:, first_panel], color = 'black', linewidth = 1)
    ax[0].grid() 
     
    ax[1].plot(Data[-window:, second_panel], color = 'brown', linewidth = 1)
    ax[1].grid()
    ax[1].legend()
# my_data = pd.read_excel('VIX.xlsx')
my_data = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix-1m.pkl")
my_data = np.array(my_data)

In [ ]:
def signal(Data, vix, buy):
    
    Data = adder(Data, 1)
    
    for i in range(len(Data)):
        
        if Data[i, vix] >= 35 and Data[i - 1, buy] == 0 and Data[i - 2, buy] == 0 and Data[i - 3, buy] == 0 and Data[i - 4, buy] == 0 and Data[i - 5, buy] == 0:
            
            Data[i, buy] = 1
            
    return Data

In [ ]:
def signal_chart_ohlc_color(Data, what_bull, window = 1000):   
     
    Plottable = Data[-window:, ]
    
    fig, ax = plt.subplots(figsize = (10, 5))
    
    plt.plot(Data[-window:, 1], color = 'black', linewidth = 1)
    for i in range(len(Plottable)):
        
        if Plottable[i, 2] == 1:
            
            x = i
            y = Plottable[i, 1]
        
            ax.annotate(' ', xy = (x, y), arrowprops = dict(width = 9, headlength = 11, headwidth = 11, facecolor = 'green', color = 'green'))    
        ax.set_facecolor((0.95, 0.95, 0.95)) 
        plt.legend() 
        plt.grid()

In [ ]:
# Choosing an example of 21 periods
period = 21
def signal_quality(Data, closing, buy, period, where):
    
    Data = adder(Data, 1)
    
    for i in range(len(Data)):
        
        try:   
            
            if Data[i, buy] == 1:
                
                Data[i + period, where] = Data[i + period, closing] - Data[i, closing]
                            
        except IndexError:
            
             pass
         
    return Data
# Using 21 Periods as a Window of signal Quality Check
my_data = signal_quality(my_data, 1, 2, period, 3)
positives = my_data[my_data[:, 3] > 0]
negatives = my_data[my_data[:, 3] < 0]
# Calculating Signal Quality
signal_quality = len(positives) / (len(negatives) + len(positives))
print('Signal Quality = ', round(signal_quality * 100, 2), '%')
# Output: Signal Quality = 86.67%

In [ ]:
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
import getFamaFrenchFactors as gff

ticker = 'SPY'
start = '2020-05-01'
end = '2022-08-01'
vix = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/vix-1m.pkl")

ff3_monthly = gff.famaFrench3Factor(frequency='m')#famaFrench3Factor(frequency='m')
ff3_monthly.rename(columns={"date_ff_factors": 'Date'}, inplace=True)
ff3_monthly.set_index('Date', inplace=True)

In [ ]:
ff_data.index = pd.to_datetime(vix.index, format = '%m/%d/%Y').strftime('%Y%m%d')
stock_returns = vix['Close'].resample('M').last().pct_change().dropna()
stock_returns.name = "Month_Rtn"
ff_data = ff3_monthly.merge(stock_returns,on='Time')

In [ ]:
ff_data.index = pd.to_datetime(ff_data.index, format = '%m/%d/%Y').strftime('%Y%m%d')

In [ ]:
ff_data.to_csv("./CSV/FamaFrenchVIX.csv")

In [ ]:
X = ff_data[['Mkt-RF', 'SMB', 'HML']]
y = ff_data['Month_Rtn'] - ff_data['RF']
X = sm.add_constant(X)
ff_model = sm.OLS(y, X).fit()
print(ff_model.summary())
intercept, b1, b2, b3 = ff_model.params

In [ ]:
rf = ff_data['RF'].mean()
market_premium = ff3_monthly['Mkt-RF'].mean()
size_premium = ff3_monthly['SMB'].mean()
value_premium = ff3_monthly['HML'].mean()

expected_monthly_return = rf + b1 * market_premium + b2 * size_premium + b3 * value_premium 
expected_yearly_return = expected_monthly_return * 12
print("Expected yearly return: " + str(expected_yearly_return))